Data loading:

In [1]:
import json
import pandas as pd
import numpy as np
import re
from collections import Counter

def squad2_to_df(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        squad = json.load(f)

    rows = []
    for article in squad["data"]:
        title = article["title"]
        for para_i, para in enumerate(article["paragraphs"]):
            context = para["context"]
            paragraph_id = f"{title}__{para_i}"

            for qa in para["qas"]:
                qid = qa["id"]
                question = qa["question"]
                is_impossible = qa.get("is_impossible", False)

                can_answer = "yes" if not is_impossible else "no"

                if can_answer == "yes" and len(qa.get("answers", [])) > 0:
                    answer_text = qa["answers"][0]["text"]
                    answer_start = qa["answers"][0]["answer_start"]
                else:
                    answer_text = "NO_ANSWER"
                    answer_start = -1

                rows.append({
                    "id": qid,
                    "title": title,
                    "paragraph_id": paragraph_id,
                    "context": context,
                    "question": question,
                    "can_answer": can_answer,
                    "answer_text": answer_text,
                    "answer_start": answer_start
                })

    return pd.DataFrame(rows)


def add_basic_features(df):
    df = df.copy()

    if "can_answer" in df.columns:
        df["y"] = (df["can_answer"].astype(str).str.lower() == "yes").astype(int)
    elif "is_impossible" in df.columns:
        df["y"] = (df["is_impossible"] == False).astype(int)
    else:
        df["y"] = (df["answer_text"] != "NO_ANSWER").astype(int)

    df["q_len_char"] = df["question"].astype(str).str.len()
    df["c_len_char"] = df["context"].astype(str).str.len()
 
    df["q_len_tok"] = df["question"].astype(str).str.split().str.len()
    df["c_len_tok"] = df["context"].astype(str).str.split().str.len()
    
    # answer len (solo se presente)
    if "answer_text" in df.columns:
        df["a_len_char"] = df["answer_text"].astype(str).str.len()
        df["a_len_tok"] = df["answer_text"].astype(str).str.split().str.len()
        df.loc[df["answer_text"] == "NO_ANSWER", ["a_len_char","a_len_tok"]] = 0
    
    return df


train_df = squad2_to_df("train_sampled.json")
val_df   = squad2_to_df("val_sampled.json")
test_df  = squad2_to_df("test_sampled.json")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

train_df = add_basic_features(train_df)
val_df   = add_basic_features(val_df)
test_df  = add_basic_features(test_df)

Train shape: (8001, 8)
Validation shape: (996, 8)
Test shape: (1004, 8)


Importing DeBERTA v3 base model:

In [2]:
pip install transformers datasets evaluate

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install accelerate

Note: you may need to restart the kernel to use updated packages.


In [6]:
import accelerate
print(accelerate.__version__)

1.13.0


/opt/anaconda3/envs/tf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We use PyTorch since DeBERTa is implemented in Pytorch:

In [7]:
import torch
import tiktoken
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer
)
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu" # we are using GPU if available, otherwise CPU
)

print(device)

mps


In [8]:
model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForQuestionAnswering.from_pretrained(model_name)
model.to(device)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 40540.50it/s]
DebertaV2ForQuestionAnswering LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
qa_outputs.weight                       | MISSING    | 
qa_outputs.bias                         | MISSING    | 

Notes:
- UNEXPE

DebertaV2ForQuestionAnswering(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm(

Tokenization and preprocessing:

In [9]:

train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

max_length = 384

def preprocess(example):

    tokenized = tokenizer(
        example["question"],
        example["context"],
        truncation="only_second",
        max_length=max_length,
        padding="max_length",
        return_offsets_mapping=True
    )

    offsets = tokenized.pop("offset_mapping")

    start_char = example["answer_start"]
    answer_text = example["answer_text"]

    if answer_text == "NO_ANSWER":
        tokenized["start_positions"] = 0
        tokenized["end_positions"] = 0
        return tokenized

    end_char = start_char + len(answer_text)

    start_token = 0
    end_token = 0

    for i, (start, end) in enumerate(offsets):
        if start <= start_char < end:
            start_token = i
        if start < end_char <= end:
            end_token = i

    tokenized["start_positions"] = start_token
    tokenized["end_positions"] = end_token

    return tokenized

In [10]:
train_dataset = train_dataset.map(preprocess)
val_dataset   = val_dataset.map(preprocess)

Map: 100%|██████████| 996/996 [00:00<00:00, 2495.01 examples/s]


Conversion in Pytorch dataset:

In [11]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","start_positions","end_positions"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","start_positions","end_positions"]
)

DataLoader:

In [12]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=4)

Training configuration

In [19]:
training_args = TrainingArguments(
    output_dir="./deberta_squad",

    learning_rate=1e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100,

    warmup_steps=int(0.2 * len(train_dataset)),

    save_steps=500,
    save_total_limit=2,

    fp16=False,
    bf16=False
)

In [14]:
from transformers import default_data_collator
from transformers import EarlyStoppingCallback

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2
)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[early_stopping]
)

In [62]:
train_dataset[0].keys()

dict_keys(['input_ids', 'attention_mask', 'start_positions', 'end_positions'])

TRAINING LOOP

In [67]:
trainer.train()

Step,Training Loss
100,3.780070
200,3.022920
300,2.968610
400,2.703530
500,2.532306
600,2.150943
700,1.974730
800,2.086137
900,1.862836
1000,1.775166


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.65it/s]
/opt/anaconda3/envs/tf/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.41it/s]
/opt/anaconda3/envs/tf/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.69s/it]


TrainOutput(global_step=6003, training_loss=1.3116638766474153, metrics={'train_runtime': 7855.1606, 'train_samples_per_second': 3.056, 'train_steps_per_second': 0.764, 'total_flos': 4705582293075456.0, 'train_loss': 1.3116638766474153, 'epoch': 3.0})

Evaluation:

In [68]:
predictions = trainer.predict(val_dataset)

/opt/anaconda3/envs/tf/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [76]:
import numpy as np

start_logits, end_logits = predictions.predictions

start_ind = np.argmax(start_logits, axis=1)
end_ind = np.argmax(end_logits, axis=1)

In [77]:
# to reconstruct the answer
answers = []

for i in range(len(start_ind)):
    input_ids = val_dataset[i]["input_ids"]
    answer_tokens = input_ids[start_ind[i]: end_ind[i]+1]
    answer = tokenizer.decode(answer_tokens)
    
    answers.append(answer)

In [71]:
import re
import string
from collections import Counter

# funzione per normalizzare testo
def normalize_text(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in set(string.punctuation))
    s = ' '.join(s.split())
    return s

# exact match
def compute_exact(a_gold, a_pred):
    return int(normalize_text(a_gold) == normalize_text(a_pred))

# F1 score
def compute_f1(a_gold, a_pred):

    gold_tokens = normalize_text(a_gold).split()
    pred_tokens = normalize_text(a_pred).split()

    common = Counter(gold_tokens) & Counter(pred_tokens)
    num_same = sum(common.values())

    if len(gold_tokens) == 0 or len(pred_tokens) == 0:
        return int(gold_tokens == pred_tokens)

    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    f1 = (2 * precision * recall) / (precision + recall)

    return f1

In [78]:
exact_scores = []
f1_scores = []

for i in range(len(val_dataset)):

    input_ids = val_dataset[i]["input_ids"]

    start = start_ind[i]
    end = end_ind[i]

    # predizione modello
    if end < start:
        pred_answer = ""
    else:
        pred_answer = tokenizer.decode(
            input_ids[start:end+1],
            skip_special_tokens=True
        )

    # risposta vera
    gold_answer = tokenizer.decode(
        input_ids[
            val_dataset[i]["start_positions"]:
            val_dataset[i]["end_positions"]+1
        ],
        skip_special_tokens=True
    )

    exact_scores.append(compute_exact(gold_answer, pred_answer))
    f1_scores.append(compute_f1(gold_answer, pred_answer))

In [79]:
EM = np.mean(exact_scores) * 100
F1 = np.mean(f1_scores) * 100

print("Exact Match:", EM)
print("F1 Score:", F1)

Exact Match: 65.46184738955823
F1 Score: 72.19484026170522


Evaluation with the evaluation metrics already available on Squad 2.0

In [80]:
import numpy as np
import evaluate

# carica metrica SQuAD
metric = evaluate.load("squad_v2")

# ottieni logits dal modello
predictions = trainer.predict(val_dataset)

start_logits, end_logits = predictions.predictions

start_indexes = np.argmax(start_logits, axis=1)
end_indexes = np.argmax(end_logits, axis=1)

formatted_predictions = []
references = []

for i in range(len(val_dataset)):

    input_ids = val_dataset[i]["input_ids"]

    start = start_indexes[i]
    end = end_indexes[i]

    # evitare span invalidi
    if end < start:
        answer = ""
    else:
        answer_tokens = input_ids[start:end+1]
        answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)

    example_id = str(i)

    formatted_predictions.append({
        "id": example_id,
        "prediction_text": answer,
        "no_answer_probability": 0.0
    })

    # ground truth
    ref_answer = tokenizer.decode(
        input_ids[
            val_dataset[i]["start_positions"]:
            val_dataset[i]["end_positions"] + 1
        ],
        skip_special_tokens=True
    )

    references.append({
        "id": example_id,
        "answers": {
            "text": [ref_answer],
            "answer_start": [0]
        }
    })

# calcolo metriche
results = metric.compute(
    predictions=formatted_predictions,
    references=references
)


/opt/anaconda3/envs/tf/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'exact': 65.46184738955823, 'f1': 72.19484026170525, 'total': 996, 'HasAns_exact': 65.46184738955823, 'HasAns_f1': 72.19484026170525, 'HasAns_total': 996, 'best_exact': 65.46184738955823, 'best_exact_thresh': 0.0, 'best_f1': 72.19484026170525, 'best_f1_thresh': 0.0}


In [81]:
results

{'exact': 65.46184738955823,
 'f1': 72.19484026170525,
 'total': 996,
 'HasAns_exact': 65.46184738955823,
 'HasAns_f1': 72.19484026170525,
 'HasAns_total': 996,
 'best_exact': 65.46184738955823,
 'best_exact_thresh': 0.0,
 'best_f1': 72.19484026170525,
 'best_f1_thresh': 0.0}